# FARL Healer LoRA — Colab Training Notebook

**Pipeline:**
1. Upload `taxonomy.json` (from `farl_hunt.py` run locally)
2. Generate corrected reasoning chains for each failure (Claude API as oracle)
3. Format as instruction-following SFT dataset
4. QLoRA fine-tune `mistralai/Mistral-7B-Instruct-v0.2` on T4 (~2hr) or `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (~20min)
5. Save adapter to Google Drive
6. Evaluate: re-score failed questions → measure failure rate reduction

**Requirements:**
- Colab T4 (free tier works for TinyLlama; Mistral-7B needs T4 with QLoRA)
- `ANTHROPIC_API_KEY` for oracle chain generation (Haiku, ~$0.02 for 20 chains)
- Google Drive mounted (to save adapter weights)

**Cost estimate:** $0.02 oracle + $0 Colab free GPU

In [ ]:
# @title Step 0: Choose model tier
# @markdown **T4 free (recommended):** TinyLlama-1.1B — fast, fits easily, good for testing the pipeline
# @markdown **T4 free (Mistral):** Mistral-7B-Instruct with QLoRA — the actual FARL healer, ~2hr
# @markdown **A100 (Colab Pro):** Mistral-7B full LoRA — ~30min, highest quality

MODEL_TIER = "tinyllama"  # @param ["tinyllama", "mistral-7b-qlora", "mistral-7b-lora"]
ANTHROPIC_API_KEY = ""    # @param {type:"string"}
USE_DRIVE = True           # @param {type:"boolean"}

MODEL_MAP = {
    "tinyllama":        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "mistral-7b-qlora": "mistralai/Mistral-7B-Instruct-v0.2",
    "mistral-7b-lora":  "mistralai/Mistral-7B-Instruct-v0.2",
}
USE_4BIT = MODEL_TIER in ("mistral-7b-qlora",)
MODEL_ID  = MODEL_MAP[MODEL_TIER]

print(f"Model:   {MODEL_ID}")
print(f"4-bit:   {USE_4BIT}")
print(f"Drive:   {USE_DRIVE}")

In [ ]:
# @title Step 1: Install dependencies
!pip install -q transformers peft trl bitsandbytes accelerate datasets anthropic llm-guard-kit
print("Dependencies installed.")

In [ ]:
# @title Step 2: Mount Google Drive (saves adapter weights)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/farl_healer_adapter"
else:
    SAVE_DIR = "/content/farl_healer_adapter"

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Adapter will be saved to: {SAVE_DIR}")

In [ ]:
# @title Step 3: Upload taxonomy.json from local machine
# @markdown Run farl_hunt.py locally first:
# @markdown `python3 scripts/farl_hunt.py --n 50 --budget 2.0`
# @markdown Then upload `results/farl_hunt/taxonomy.json` here.

import json
from google.colab import files

print("Upload your taxonomy.json file:")
uploaded = files.upload()

if 'taxonomy.json' in uploaded:
    taxonomy = json.loads(uploaded['taxonomy.json'])
else:
    # Try first uploaded file
    fname = list(uploaded.keys())[0]
    taxonomy = json.loads(uploaded[fname])

# Flatten all failures into a list
all_failures = []
for mode, entries in taxonomy.items():
    for entry in entries:
        entry['failure_mode'] = mode
        all_failures.append(entry)

print(f"Loaded {len(all_failures)} novel failures from taxonomy:")
for mode, entries in taxonomy.items():
    if entries:
        print(f"  {mode}: {len(entries)} failures")

In [ ]:
# @title Step 4: Generate oracle corrections (Claude Haiku as teacher)
# @markdown For each failed question, we ask Claude to produce the CORRECT reasoning chain.
# @markdown This is the SFT training signal: (wrong_chain) → (correct_chain)
# @markdown Cost: ~$0.001/question × 50 = ~$0.05

import anthropic
import time

ORACLE_SYSTEM = """You are an expert research assistant. You will be shown a question that 
caused a ReAct reasoning agent to fail. Produce the CORRECT step-by-step reasoning chain 
to answer this question accurately.

Format your response EXACTLY as:
Thought: <reasoning>
Action: Search[<specific query>]
Observation: <what you would find>
... (repeat 2-4 times)
Thought: <final reasoning>
Action: Finish[<precise answer>]

Be accurate. Use specific, targeted search queries. Don't repeat queries."""

def generate_oracle_chain(question: str, failed_answer: str, client) -> str:
    user = f"""Question: {question}

The agent previously failed with this wrong answer: {failed_answer}

Produce the correct reasoning chain:"""
    
    for attempt in range(3):
        try:
            resp = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=600,
                system=ORACLE_SYSTEM,
                messages=[{"role": "user", "content": user}],
            )
            return resp.content[0].text
        except Exception as e:
            if attempt == 2: raise
            time.sleep(2 ** attempt)

if not ANTHROPIC_API_KEY:
    print("WARNING: No ANTHROPIC_API_KEY — skipping oracle generation.")
    print("Set ANTHROPIC_API_KEY in Step 0 to generate corrected chains.")
    oracle_corrections = []
else:
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    oracle_corrections = []
    
    print(f"Generating oracle corrections for {len(all_failures)} failures...")
    for i, failure in enumerate(all_failures):
        question     = failure['question']
        failed_ans   = failure.get('final_answer', '')
        failure_mode = failure.get('failure_mode', 'unknown')
        
        print(f"  [{i+1}/{len(all_failures)}] {failure_mode}: {question[:60]}...")
        correction = generate_oracle_chain(question, failed_ans, client)
        oracle_corrections.append({
            'question':     question,
            'failed_answer': failed_ans,
            'failure_mode': failure_mode,
            'oracle_chain': correction,
        })
        time.sleep(0.3)  # rate limit
    
    print(f"\nGenerated {len(oracle_corrections)} corrections.")
    
    # Save corrections
    with open('/content/oracle_corrections.json', 'w') as f:
        json.dump(oracle_corrections, f, indent=2)
    print("Saved to /content/oracle_corrections.json")

In [ ]:
# @title Step 5: Format SFT dataset
# @markdown Creates instruction-following pairs:
# @markdown  Input:  system_prompt + question
# @markdown  Output: correct reasoning chain (oracle)

from datasets import Dataset

HEALER_SYSTEM = """You are a careful research assistant using the ReAct framework.
Answer questions by searching step by step. Use specific, targeted queries.
Never repeat the same search. Stop and give a precise final answer when confident.

Format:
Thought: <reasoning>
Action: Search[<query>] or Action: Finish[<answer>]"""

def format_sample(correction: dict, tokenizer) -> dict:
    """Format as chat template for instruction fine-tuning."""
    messages = [
        {"role": "system",    "content": HEALER_SYSTEM},
        {"role": "user",      "content": f"Question: {correction['question']}"},
        {"role": "assistant", "content": correction['oracle_chain']},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

if not oracle_corrections:
    # Fallback: create synthetic examples if no oracle
    print("No oracle corrections available — using synthetic fallback examples.")
    oracle_corrections = [
        {
            'question': f['question'],
            'failed_answer': f.get('final_answer', ''),
            'failure_mode': f.get('failure_mode', 'unknown'),
            'oracle_chain': (
                f"Thought: I need to search carefully for the answer.\n"
                f"Action: Search[{f['question'][:50]}]\n"
                f"Observation: [factual information about this topic]\n"
                f"Thought: Based on the search, I can now answer.\n"
                f"Action: Finish[The answer based on the search result]"
            ),
        }
        for f in all_failures
    ]

print(f"SFT dataset: {len(oracle_corrections)} examples")
for ex in oracle_corrections[:2]:
    print(f"\n  Q: {ex['question'][:70]}")
    print(f"  Mode: {ex['failure_mode']}")
    print(f"  Oracle chain ({len(ex['oracle_chain'])} chars)")

In [ ]:
# @title Step 6: Load model + apply LoRA
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

print(f"Loading {MODEL_ID} ...")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Model (4-bit for Mistral, full precision for TinyLlama)
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
    )

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                     # rank
    lora_alpha=32,            # scaling
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # attention projections only
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model ready.")

In [ ]:
# @title Step 7: Prepare dataset and train
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Format all examples
texts = [format_sample(c, tokenizer) for c in oracle_corrections]
dataset = Dataset.from_dict({"text": texts})
print(f"Training on {len(dataset)} examples")
print(f"Example:\n{texts[0][:300]}...")

# Training config
# Small dataset (20-50 examples) → 5 epochs, small batch
n_examples = len(dataset)
n_epochs   = min(10, max(3, 150 // n_examples))  # ~150 total steps
print(f"\nTraining: {n_epochs} epochs × {n_examples} examples = {n_epochs*n_examples} total steps")

import trl as _trl
_trl_version = tuple(int(x) for x in _trl.__version__.split('.')[:2])

_sft_config_kwargs = dict(
    output_dir=SAVE_DIR,
    num_train_epochs=n_epochs,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    optim="adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="none",
)
# max_seq_length and dataset_text_field moved to SFTTrainer in trl>=0.9
if _trl_version < (0, 9):
    _sft_config_kwargs["max_seq_length"] = 1024
    _sft_config_kwargs["dataset_text_field"] = "text"

training_args = SFTConfig(**_sft_config_kwargs)

import trl as _trl
_trl_version = tuple(int(x) for x in _trl.__version__.split('.')[:2])
_trainer_kwargs = {'processing_class': tokenizer} if _trl_version >= (0, 9) else {'tokenizer': tokenizer}
# max_seq_length moved from SFTConfig to SFTTrainer in trl>=0.9
if _trl_version >= (0, 9):
    _trainer_kwargs['max_seq_length'] = 1024
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    **_trainer_kwargs,
)

print("\nStarting LoRA fine-tuning...")
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"  Loss:    {train_result.training_loss:.4f}")
print(f"  Steps:   {train_result.global_step}")

In [ ]:
# @title Step 8: Save adapter
adapter_path = os.path.join(SAVE_DIR, "farl_healer_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

# Save metadata
metadata = {
    "base_model": MODEL_ID,
    "model_tier": MODEL_TIER,
    "n_training_examples": len(oracle_corrections),
    "failure_modes_covered": list(taxonomy.keys()),
    "training_loss": train_result.training_loss,
    "n_epochs": n_epochs,
}
with open(os.path.join(adapter_path, 'farl_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Adapter saved to: {adapter_path}")
print(f"  Base model: {MODEL_ID}")
print(f"  Trained on: {len(oracle_corrections)} examples")
print(f"  Modes: {list(taxonomy.keys())}")

# List saved files
for f in os.listdir(adapter_path):
    size = os.path.getsize(os.path.join(adapter_path, f))
    print(f"  {f}: {size/1024:.1f} KB")

In [ ]:
# @title Step 9: Evaluate — score failed questions before vs after LoRA
# @markdown Measures how much the LoRA adapter reduces failure rate on the taxonomy questions.

import re
from peft import PeftModel

def run_healed_chain(question: str, model, tokenizer, max_steps: int = 6) -> dict:
    """Run ReAct chain with the LoRA-patched model (no API needed)."""
    messages = [
        {"role": "system", "content": HEALER_SYSTEM},
        {"role": "user",   "content": f"Question: {question}"},
    ]
    steps = []
    conversation = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    for _ in range(max_steps):
        inputs = tokenizer(conversation, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        # Parse action
        finish_match = re.search(r"Action:\s*Finish\[(.+?)\]", response)
        search_match = re.search(r"Action:\s*Search\[(.+?)\]", response)
        thought_match = re.search(r"Thought:\s*(.+?)(?=Action:|$)", response, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else ""
        
        if finish_match:
            answer = finish_match.group(1).strip()
            steps.append({"action_type": "Finish", "action_arg": answer, "thought": thought, "observation": ""})
            return {"final_answer": answer, "steps": steps, "finished": True}
        
        query = search_match.group(1).strip() if search_match else response[:50]
        steps.append({"action_type": "Search", "action_arg": query, "thought": thought, "observation": "[search result]"})
        conversation += response + f"\nObservation: [factual result for '{query}']\n"
    
    return {"final_answer": "", "steps": steps, "finished": False}

# Score same questions with healed model
from llm_guard import AgentGuard
guard = AgentGuard()

print("Evaluating healed model on taxonomy questions...")
before_risks = [f['risk_score'] for f in all_failures]
after_risks  = []
after_stuck  = []

for i, failure in enumerate(all_failures[:20]):  # eval on first 20
    q = failure['question']
    chain = run_healed_chain(q, model, tokenizer)
    result = guard.score_chain(q, chain['steps'], chain['final_answer'])
    after_risks.append(result.risk_score)
    stuck = not chain['finished'] and len(chain['steps']) >= 5
    after_stuck.append(stuck)
    print(f"  [{i+1}] risk: {failure['risk_score']:.3f} → {result.risk_score:.3f}  stuck: {stuck}")

import numpy as np
print(f"\n{'='*50}")
print(f"EVALUATION RESULTS")
print(f"{'='*50}")
print(f"  Mean risk BEFORE:  {np.mean(before_risks[:20]):.3f}")
print(f"  Mean risk AFTER:   {np.mean(after_risks):.3f}")
print(f"  Risk reduction:    {np.mean(before_risks[:20]) - np.mean(after_risks):.3f}")
print(f"  Stuck chains AFTER: {sum(after_stuck)}/{len(after_stuck)}")

In [ ]:
# @title Step 10: Retrain MiniJudge on taxonomy (runs on CPU, no GPU needed)
# @markdown This is the fastest improvement path.
# @markdown Uses the taxonomy wrong chains + synthetic correct chains to update MiniJudge.
# @markdown Result: improved AUROC on adversarial failure patterns.

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from llm_guard.local_verifier import extract_features, FEATURE_NAMES

# Wrong examples from taxonomy
X_wrong, y_wrong = [], []
for failure in all_failures:
    steps = failure.get('step_details', [])
    if not steps:
        continue
    feat = extract_features(failure['question'], steps, failure['final_answer'])
    X_wrong.append(feat)
    y_wrong.append(1)  # label=1 means WRONG

# Synthetic correct examples (simple 2-step clean chains)
CORRECT_TEMPLATE = [
    {"thought": "I need to search for this.", "action_type": "Search",
     "action_arg": "query", "observation": "Relevant factual information found."},
    {"thought": "I have the answer.", "action_type": "Finish",
     "action_arg": "correct answer", "observation": ""},
]
correct_questions = [
    "What year was the Eiffel Tower built?",
    "Who wrote Romeo and Juliet?",
    "What is the capital of France?",
    "How many planets are in the solar system?",
    "Who painted the Mona Lisa?",
]
X_correct, y_correct = [], []
for q in correct_questions:
    feat = extract_features(q, CORRECT_TEMPLATE, "correct answer")
    X_correct.append(feat)
    y_correct.append(0)  # label=0 means CORRECT

# Combine
if X_wrong and X_correct:
    X = np.vstack(X_wrong + X_correct)
    y = np.array(y_wrong + y_correct)
    print(f"Training MiniJudge update: {sum(y==1)} wrong + {sum(y==0)} correct")
    
    # Train updated LogReg
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000)),
    ])
    pipe.fit(X, y)
    
    # Save
    import joblib
    judge_path = os.path.join(SAVE_DIR, 'mini_judge_updated.pkl')
    joblib.dump(pipe, judge_path)
    print(f"Saved updated MiniJudge to: {judge_path}")
    
    # Quick eval: score wrong vs correct
    preds_wrong   = pipe.predict_proba(np.vstack(X_wrong))[:, 1]
    preds_correct = pipe.predict_proba(np.vstack(X_correct))[:, 1]
    print(f"  Mean risk (wrong):   {preds_wrong.mean():.3f}")
    print(f"  Mean risk (correct): {preds_correct.mean():.3f}")
    print(f"  Separation:          {preds_wrong.mean() - preds_correct.mean():.3f}")
else:
    print("Not enough data for MiniJudge update. Run more iterations of farl_hunt.")

## What to do with the results

After training completes:

1. **Download `farl_healer_adapter/` from Drive** — the LoRA adapter weights
2. **Download `mini_judge_updated.pkl` from Drive** — drop into `llm_guard/data/mini_judge.pkl` to ship with next PyPI version
3. **Run evaluation**: Score the taxonomy questions with the new judge vs old judge — measure AUROC improvement
4. **Next step**: Run `farl_hunt.py --n 200 --resume` to collect more taxonomy data and repeat the loop

### The full FARL loop (CPU + Colab)

```
Local (CPU, $0.07):                Colab (GPU, $0 free tier):
farl_hunt.py --n 50           →   Upload taxonomy.json
  ↓                                 ↓
taxonomy.json                    Run this notebook
  (novel failures)                  ↓
                                 mini_judge_updated.pkl
                                 farl_healer_adapter/
                                    ↓
Drop updated pkl into            Download and replace
llm_guard/data/mini_judge.pkl    llm_guard/data/mini_judge.pkl
  ↓                                  
Bump to v0.18.0 → PyPI
```